# Edge Latency and Quantization Benchmark

This notebook benchmarks the selected deployment model, `baseline_yolo26n`, for edge deployment. It compares ONNX FP32, FP16, and INT8 variants for model size, latency, and violation-centric detection metrics.

## 1. Controls

Edit these values first. `FAST_DEBUG=True` uses fewer validation images so the notebook can verify the full path quickly before a full benchmark.

In [ ]:
!pip install ultralytics onnxconverter-common onnxruntime

In [ ]:
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
OUTPUT_ROOT = KAGGLE_WORKING / 'edge_latency_quantization'
RESULTS_DIR = KAGGLE_WORKING / 'results'
EXPORTS_DIR = KAGGLE_WORKING / 'exports'
CONFIGS_DIR = KAGGLE_WORKING / 'configs'

MODEL_ONNX_PATH = None
MODEL_PT_PATH = None
DATASET_ROOT = None
DEVICE = None

FAST_DEBUG = True
SEED = 42
IMGSZ = 640
DEBUG_MAX_IMAGES = 64
BENCHMARK_MAX_IMAGES = 512
BATCH_SIZES = [1, 4, 8]
WARMUP_RUNS = 10
TIMED_RUNS = 50
CREATE_DYNAMIC_ONNX_FROM_PT = True
RUN_FP16_EXPORT_FROM_PT = True
RUN_INT8_DYNAMIC = True
ACCEPTABLE_MAP_DROP = 0.02
VIOLATION_CLASS_IDS = [7, 8, 9, 10]

CLASS_NAMES = {
    0: 'helmet', 1: 'gloves', 2: 'vest', 3: 'boots', 4: 'goggles',
    5: 'none', 6: 'Person', 7: 'no_helmet', 8: 'no_goggle',
    9: 'no_gloves', 10: 'no_boots',
}

for d in [OUTPUT_ROOT, RESULTS_DIR, EXPORTS_DIR, CONFIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('FAST_DEBUG:', FAST_DEBUG)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

## 2. Environment and GPU Check

This records package versions and GPU visibility. Kaggle T4 x2 should expose two CUDA devices; ONNXRuntime latency is still measured directly through available execution providers.

In [ ]:
import json
import os
import random
import shutil
import subprocess
import sys
import time
from collections import Counter

import numpy as np
import pandas as pd

def version_of(import_name):
    try:
        mod = __import__(import_name)
        return getattr(mod, '__version__', 'installed-version-unknown')
    except Exception as exc:
        return f'not installed ({exc.__class__.__name__})'

print('Python:', sys.version)
for name in ['torch', 'ultralytics', 'onnxruntime', 'onnx', 'onnxconverter_common', 'cv2']:
    print(f'{name}:', version_of(name))

try:
    import torch
    cuda_count = torch.cuda.device_count()
    DEVICE = '0,1' if cuda_count >= 2 else ('0' if cuda_count == 1 else 'cpu') if DEVICE is None else DEVICE
    print('torch.cuda.device_count():', cuda_count)
    for idx in range(cuda_count):
        print(f'GPU {idx}:', torch.cuda.get_device_name(idx))
except Exception as exc:
    DEVICE = DEVICE or 'cpu'
    print('Torch GPU check failed:', repr(exc))

try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except Exception as exc:
    print('nvidia-smi unavailable:', repr(exc))

print('Selected DEVICE:', DEVICE)

## 3. Locate Dataset and Baseline Model

The notebook searches `/kaggle/input` and `/kaggle/working` for the packaged `baseline_yolo26n` ONNX and optional PyTorch weights.

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def has_dataset_layout(path):
    p = Path(path)
    return all((p / 'images' / s).exists() and (p / 'labels' / s).exists() for s in ['train', 'val', 'test'])

def find_dataset_root():
    if DATASET_ROOT is not None:
        return Path(DATASET_ROOT)
    for base in [KAGGLE_INPUT, KAGGLE_WORKING, Path.cwd()]:
        if not base.exists():
            continue
        for candidate in [base] + [p for p in base.rglob('*') if p.is_dir() and p.name.lower() in {'merged-ppe', 'dataset', 'data', 'ppe'}]:
            if has_dataset_layout(candidate):
                return candidate.resolve()
    raise FileNotFoundError('Set DATASET_ROOT manually. Could not find images/ and labels/ splits.')

def find_first(patterns):
    roots = [KAGGLE_WORKING, KAGGLE_INPUT, Path.cwd()]
    for root in roots:
        if not root.exists():
            continue
        for pattern in patterns:
            matches = sorted(root.rglob(pattern))
            if matches:
                return matches[0].resolve()
    return None

DATASET_ROOT = find_dataset_root()
MODEL_ONNX_PATH = Path(MODEL_ONNX_PATH) if MODEL_ONNX_PATH else find_first(['baseline_yolo26n_best.onnx', 'baseline_yolo26n/weights/best.onnx', 'best.onnx'])
MODEL_PT_PATH = Path(MODEL_PT_PATH) if MODEL_PT_PATH else find_first(['baseline_yolo26n_best.pt', 'baseline_yolo26n/weights/best.pt', 'best.pt'])

if MODEL_ONNX_PATH is None or not MODEL_ONNX_PATH.exists():
    raise FileNotFoundError('Could not find baseline_yolo26n ONNX. Attach the package or set MODEL_ONNX_PATH.')

WRITABLE_MODEL_PT_PATH = None
if MODEL_PT_PATH is not None and Path(MODEL_PT_PATH).exists():
    WRITABLE_MODEL_PT_PATH = EXPORTS_DIR / 'baseline_yolo26n_best_for_export.pt'
    shutil.copy2(MODEL_PT_PATH, WRITABLE_MODEL_PT_PATH)
    print('Copied PyTorch weights to writable export path:', WRITABLE_MODEL_PT_PATH)

print('DATASET_ROOT:', DATASET_ROOT)
print('MODEL_ONNX_PATH:', MODEL_ONNX_PATH)
print('MODEL_PT_PATH:', MODEL_PT_PATH)
print('WRITABLE_MODEL_PT_PATH:', WRITABLE_MODEL_PT_PATH)

## 4. Create Kaggle Data YAML

This YAML keeps the original split semantics and class names. No dataset files are modified.

In [ ]:
def write_data_yaml(path, dataset_root):
    lines = [f'path: {dataset_root}', 'train: images/train', 'val: images/val', 'test: images/test', 'names:']
    for idx, name in CLASS_NAMES.items():
        lines.append(f'  {idx}: {name}')
    path.write_text('\n'.join(lines) + '\n')
    return path

DATA_YAML = write_data_yaml(CONFIGS_DIR / 'data_edge_latency.yaml', DATASET_ROOT)
print(DATA_YAML.read_text())

## 5. Build FP32, FP16, and INT8 ONNX Variants

FP32 fixed-batch is the original export. If PyTorch weights are available, the notebook also exports a dynamic-batch FP32 ONNX for batch-size experiments. FP16 is exported from PyTorch weights with Ultralytics instead of generic graph conversion because generic FP16 conversion can break YOLO `Resize` node dtypes. INT8 dynamic quantization is kept as an empirical test and can be rejected if it is slower than FP32.

In [ ]:
import onnxruntime as ort
from ultralytics import YOLO

VARIANTS = []

def providers_for_validation():
    available = ort.get_available_providers()
    providers = []
    if 'CUDAExecutionProvider' in available:
        providers.append('CUDAExecutionProvider')
    providers.append('CPUExecutionProvider')
    return providers

def validate_onnx_session(path):
    session = ort.InferenceSession(str(path), providers=providers_for_validation())
    input_meta = session.get_inputs()[0]
    return {'input_shape': list(input_meta.shape), 'input_type': input_meta.type}

def add_variant(name, path, status, note, source=''):
    meta = {'input_shape': None, 'input_type': None}
    if status == 'ok' and Path(path).exists():
        try:
            meta = validate_onnx_session(path)
        except Exception as exc:
            status = 'skipped'
            note = f'ONNXRuntime validation failed: {exc!r}'
    item = {'variant': name, 'path': Path(path), 'status': status, 'note': note, 'source': source}
    item.update(meta)
    VARIANTS.append(item)

fp32_path = EXPORTS_DIR / 'baseline_yolo26n_fp32_fixed_b1.onnx'
shutil.copy2(MODEL_ONNX_PATH, fp32_path)
add_variant('fp32_fixed_b1', fp32_path, 'ok', 'original ONNX export; fixed batch size is expected', source=str(MODEL_ONNX_PATH))

fp32_dynamic_path = EXPORTS_DIR / 'baseline_yolo26n_fp32_dynamic.onnx'
if CREATE_DYNAMIC_ONNX_FROM_PT and WRITABLE_MODEL_PT_PATH is not None and Path(WRITABLE_MODEL_PT_PATH).exists():
    try:
        model = YOLO(str(WRITABLE_MODEL_PT_PATH))
        exported = Path(model.export(format='onnx', imgsz=IMGSZ, dynamic=True, half=False, simplify=True))
        shutil.copy2(exported, fp32_dynamic_path)
        add_variant('fp32_dynamic', fp32_dynamic_path, 'ok', 'Ultralytics export from writable best.pt copy with dynamic=True', source=str(WRITABLE_MODEL_PT_PATH))
    except Exception as exc:
        add_variant('fp32_dynamic', fp32_dynamic_path, 'skipped', f'dynamic FP32 export failed: {exc!r}', source=str(WRITABLE_MODEL_PT_PATH))
else:
    add_variant('fp32_dynamic', fp32_dynamic_path, 'skipped', 'WRITABLE_MODEL_PT_PATH unavailable or CREATE_DYNAMIC_ONNX_FROM_PT=False')

fp16_path = EXPORTS_DIR / 'baseline_yolo26n_fp16_dynamic.onnx'
if RUN_FP16_EXPORT_FROM_PT and WRITABLE_MODEL_PT_PATH is not None and Path(WRITABLE_MODEL_PT_PATH).exists():
    try:
        model = YOLO(str(WRITABLE_MODEL_PT_PATH))
        export_kwargs = dict(format='onnx', imgsz=IMGSZ, dynamic=True, half=True, simplify=True)
        if DEVICE != 'cpu':
            export_kwargs['device'] = 0
        exported = Path(model.export(**export_kwargs))
        shutil.copy2(exported, fp16_path)
        add_variant('fp16_dynamic', fp16_path, 'ok', 'Ultralytics export from writable best.pt copy with half=True and dynamic=True', source=str(WRITABLE_MODEL_PT_PATH))
    except Exception as exc:
        add_variant('fp16_dynamic', fp16_path, 'skipped', f'Ultralytics FP16 export failed: {exc!r}', source=str(WRITABLE_MODEL_PT_PATH))
else:
    add_variant('fp16_dynamic', fp16_path, 'skipped', 'WRITABLE_MODEL_PT_PATH unavailable or RUN_FP16_EXPORT_FROM_PT=False')

int8_source_path = EXPORTS_DIR / 'baseline_yolo26n_fp32_preprocessed_for_int8.onnx'
int8_shape_inferred_path = EXPORTS_DIR / 'baseline_yolo26n_fp32_shape_inferred_for_int8.onnx'
int8_path = EXPORTS_DIR / 'baseline_yolo26n_int8_dynamic.onnx'
if RUN_INT8_DYNAMIC:
    try:
        import logging
        import onnx
        from onnxruntime.quantization import QuantType, quant_pre_process, quantize_dynamic
        quant_base = fp32_dynamic_path if fp32_dynamic_path.exists() else fp32_path
        try:
            quant_pre_process(
                input_model_path=str(quant_base),
                output_model_path=str(int8_source_path),
                skip_symbolic_shape=True,
                skip_optimization=False,
                skip_onnx_shape=False,
            )
            quant_source = int8_source_path
            quant_note = 'onnxruntime quant_pre_process + dynamic quantization'
        except Exception as pre_exc:
            print('ONNXRuntime quant_pre_process failed; using ONNX shape inference fallback:', repr(pre_exc))
            model = onnx.load(str(quant_base))
            inferred = onnx.shape_inference.infer_shapes(model)
            onnx.save(inferred, str(int8_shape_inferred_path))
            quant_source = int8_shape_inferred_path
            quant_note = f'onnx shape_inference + dynamic quantization; quant_pre_process failed: {pre_exc!r}'
        root_logger = logging.getLogger()
        old_level = root_logger.level
        root_logger.setLevel(logging.ERROR)
        try:
            quantize_dynamic(str(quant_source), str(int8_path), weight_type=QuantType.QInt8)
        finally:
            root_logger.setLevel(old_level)
        add_variant('int8_dynamic', int8_path, 'ok', quant_note, source=str(quant_source))
    except Exception as exc:
        add_variant('int8_dynamic', int8_path, 'skipped', repr(exc))
else:
    add_variant('int8_dynamic', int8_path, 'skipped', 'RUN_INT8_DYNAMIC=False')

for item in VARIANTS:
    size = item['path'].stat().st_size / (1024 * 1024) if item['path'].exists() else None
    print(item['variant'], item['status'], item['path'], 'size_mb=', size, 'input_shape=', item.get('input_shape'), 'note=', item['note'])

## 6. ONNXRuntime Latency Benchmark

Latency is measured with ONNXRuntime on a fixed validation subset. The preprocessing path resizes images to `IMGSZ`, converts BGR to RGB, normalizes to `[0, 1]`, and uses NCHW layout.

In [ ]:
import cv2
import onnxruntime as ort

def load_val_images(max_images):
    paths = sorted([p for p in (DATASET_ROOT / 'images' / 'val').iterdir() if p.suffix.lower() in IMAGE_EXTS])[:max_images]
    images = []
    for path in paths:
        img = cv2.imread(str(path))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMGSZ, IMGSZ), interpolation=cv2.INTER_LINEAR)
        arr = img.astype(np.float32) / 255.0
        arr = np.transpose(arr, (2, 0, 1))
        images.append(arr)
    return np.stack(images, axis=0) if images else np.empty((0, 3, IMGSZ, IMGSZ), dtype=np.float32)

max_images = DEBUG_MAX_IMAGES if FAST_DEBUG else BENCHMARK_MAX_IMAGES
VAL_TENSOR = load_val_images(max_images)
print('Loaded validation tensor:', VAL_TENSOR.shape)

def providers_for(path):
    available = ort.get_available_providers()
    providers = []
    if 'CUDAExecutionProvider' in available:
        providers.append('CUDAExecutionProvider')
    providers.append('CPUExecutionProvider')
    return providers

def fixed_batch_size(input_shape):
    if not input_shape:
        return None
    batch_dim = input_shape[0]
    return batch_dim if isinstance(batch_dim, int) else None

def supports_batch(input_shape, batch_size):
    fixed = fixed_batch_size(input_shape)
    return fixed is None or fixed == batch_size

def benchmark_onnx(path, batch_size):
    session = ort.InferenceSession(str(path), providers=providers_for(path))
    input_meta = session.get_inputs()[0]
    input_name = input_meta.name
    input_type = input_meta.type
    input_shape = list(input_meta.shape)
    if not supports_batch(input_shape, batch_size):
        raise ValueError(f'ONNX input shape {input_shape} does not support batch_size={batch_size}')
    tensor = VAL_TENSOR
    if tensor.shape[0] == 0:
        raise RuntimeError('No validation images loaded')
    if tensor.shape[0] < batch_size:
        repeats = int(np.ceil(batch_size / tensor.shape[0]))
        tensor = np.concatenate([tensor] * repeats, axis=0)
    batch = tensor[:batch_size]
    if 'float16' in input_type:
        batch = batch.astype(np.float16)
    else:
        batch = batch.astype(np.float32)
    for _ in range(WARMUP_RUNS):
        session.run(None, {input_name: batch})
    start = time.perf_counter()
    for i in range(TIMED_RUNS):
        offset = (i * batch_size) % max(1, tensor.shape[0] - batch_size + 1)
        current = tensor[offset:offset + batch_size]
        if current.shape[0] < batch_size:
            current = batch
        current = current.astype(np.float16 if 'float16' in input_type else np.float32)
        session.run(None, {input_name: current})
    elapsed = time.perf_counter() - start
    return (elapsed * 1000.0) / (TIMED_RUNS * batch_size), session.get_providers(), input_shape, input_type

latency_rows = []
for variant in VARIANTS:
    if variant['status'] != 'ok' or not variant['path'].exists():
        continue
    for batch_size in BATCH_SIZES:
        try:
            if not supports_batch(variant.get('input_shape'), batch_size):
                latency_rows.append({
                    'variant': variant['variant'],
                    'batch_size': batch_size,
                    'latency_ms_per_img': None,
                    'fps': None,
                    'providers': '',
                    'benchmark_status': 'skipped_fixed_batch',
                    'error': f"input_shape={variant.get('input_shape')} supports only batch_size={fixed_batch_size(variant.get('input_shape'))}",
                })
                continue
            ms_per_img, providers, input_shape, input_type = benchmark_onnx(variant['path'], batch_size)
            latency_rows.append({
                'variant': variant['variant'],
                'batch_size': batch_size,
                'latency_ms_per_img': ms_per_img,
                'fps': 1000.0 / ms_per_img if ms_per_img > 0 else None,
                'providers': ';'.join(providers),
                'input_shape': str(input_shape),
                'input_type': input_type,
                'benchmark_status': 'ok',
            })
        except Exception as exc:
            latency_rows.append({'variant': variant['variant'], 'batch_size': batch_size, 'latency_ms_per_img': None, 'fps': None, 'providers': '', 'input_shape': str(variant.get('input_shape')), 'input_type': variant.get('input_type'), 'benchmark_status': 'failed', 'error': repr(exc)})

latency_df = pd.DataFrame(latency_rows)
display(latency_df)

## 7. Violation-Centric Evaluation

This attempts Ultralytics validation for each ONNX variant so that mAP and per-violation-class AP are comparable to prior experiment summaries. If a quantized model cannot be parsed by Ultralytics, the notebook records the error while preserving latency results.

In [ ]:
from ultralytics import YOLO

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

def eval_variant(variant):
    if variant['status'] != 'ok' or not variant['path'].exists():
        return {'variant': variant['variant'], 'eval_status': 'skipped', 'eval_error': variant['note']}
    try:
        model = YOLO(str(variant['path']))
        metrics = model.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ, device=DEVICE, verbose=False)
        ap50 = getattr(metrics.box, 'ap50', [])
        row = {
            'variant': variant['variant'],
            'eval_status': 'ok',
            'map50': safe_float(getattr(metrics.box, 'map50', None)),
            'map50_95': safe_float(getattr(metrics.box, 'map', None)),
            'precision': safe_float(getattr(metrics.box, 'mp', None)),
            'recall': safe_float(getattr(metrics.box, 'mr', None)),
        }
        vals = []
        for cls_id in VIOLATION_CLASS_IDS:
            val = safe_float(ap50[cls_id]) if cls_id < len(ap50) else None
            row[f'{CLASS_NAMES[cls_id]}_ap50'] = val
            if val is not None:
                vals.append(val)
        row['mean_violation_map50'] = float(np.mean(vals)) if vals else None
        return row
    except Exception as exc:
        return {'variant': variant['variant'], 'eval_status': 'failed', 'eval_error': repr(exc)}

eval_df = pd.DataFrame([eval_variant(v) for v in VARIANTS])
display(eval_df)

## 8. Save Summary

The final table joins latency, size, and violation metrics and writes `/kaggle/working/results/latency_quantization_summary.csv`.

In [ ]:
variant_meta = pd.DataFrame([
    {
        'variant': v['variant'],
        'model_path': str(v['path']),
        'model_size_mb': v['path'].stat().st_size / (1024 * 1024) if v['path'].exists() else None,
        'variant_status': v['status'],
        'variant_note': v['note'],
        'variant_input_shape': str(v.get('input_shape')),
        'variant_input_type': v.get('input_type'),
    }
    for v in VARIANTS
])
summary = latency_df.merge(variant_meta, on='variant', how='left').merge(eval_df, on='variant', how='left')

fp32_ref = summary[(summary['variant'].isin(['fp32_dynamic', 'fp32_fixed_b1'])) & (summary['batch_size'] == 1) & (summary['benchmark_status'] == 'ok')].copy()
fp32_ref = fp32_ref.sort_values('latency_ms_per_img').head(1)
ref_latency = float(fp32_ref['latency_ms_per_img'].iloc[0]) if not fp32_ref.empty else None
ref_map = float(fp32_ref['mean_violation_map50'].iloc[0]) if not fp32_ref.empty and pd.notna(fp32_ref['mean_violation_map50'].iloc[0]) else None

def recommendation(row):
    if row.get('benchmark_status') != 'ok':
        return 'not_evaluated'
    variant = str(row.get('variant'))
    latency = row.get('latency_ms_per_img')
    metric = row.get('mean_violation_map50')
    if variant.startswith('fp32'):
        return 'reference_fp32'
    if ref_latency is not None and pd.notna(latency) and float(latency) >= ref_latency:
        return 'not_recommended: slower than FP32 reference'
    if ref_map is not None and pd.notna(metric) and (ref_map - float(metric)) > ACCEPTABLE_MAP_DROP:
        return 'not_recommended: violation mAP drop too large'
    return 'candidate: faster than FP32 with acceptable metric drop'

summary['deployment_recommendation'] = summary.apply(recommendation, axis=1)
out_csv = RESULTS_DIR / 'latency_quantization_summary.csv'
out_json = RESULTS_DIR / 'latency_quantization_summary.json'
summary.to_csv(out_csv, index=False)
out_json.write_text(summary.to_json(orient='records', indent=2) + '\n')
print('Saved:', out_csv)
print('Saved:', out_json)
display(summary)